# Stickman Shot Factory — Kaggle

Runs **VS Code in the browser** on a Kaggle T4 x2 session, with the **Krea 2 Turbo Edit**
model held warm in the background so you can generate Whymentary-style stickman shots
from inside the editor.

**Before you run anything:**

| Setting | Value | Where |
|---|---|---|
| Accelerator | **GPU T4 x2** | Settings → Accelerator |
| Internet | **On** | Settings → Internet |
| Persistence | Files only | Settings → Persistence |

Two GPUs matter: the model sits on GPU0, leaving GPU1 free so a VS Code session
doing something else doesn't evict it.

**Run cells 1-5 in order**, open the printed URL, then install whatever extensions
you like from the Extensions panel. Cells 6-9 bring up the model server; you can run
them from the notebook or from a terminal inside VS Code.

Expect roughly **9 sessions/week at 12h each**. The session dies when the tab closes
for too long — cell 10 is a keepalive.

---
## 1. Preflight

Fails loudly now rather than 20 minutes in. The `T4 x2` assertion is the one that
actually bites: a P100 or a single T4 changes the memory math.

In [ ]:
import subprocess, sys, os, json, shutil

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)

import torch
n = torch.cuda.device_count()
names = [torch.cuda.get_device_name(i) for i in range(n)]
print(f"torch {torch.__version__} | cuda {torch.version.cuda} | {n} GPU(s): {names}")

# T4 is Turing (SM75): no bf16. This drives every fp16 patch in scripts/patches/.
cap = torch.cuda.get_device_capability(0)
print(f"compute capability {cap[0]}.{cap[1]}  bf16_supported={torch.cuda.is_bf16_supported()}")

assert n >= 1, "No GPU. Settings -> Accelerator -> GPU T4 x2"
if n < 2:
    print("!! Only 1 GPU visible. Everything still works, but you lose the spare.")
if cap[0] >= 8:
    print("!! Ampere+ detected: bf16 works natively, the fp16 patches are unnecessary.")

# Internet check -- Kaggle silently gives you a no-network session otherwise.
r = subprocess.run(["curl", "-sSf", "-m", "10", "-o", "/dev/null",
                    "-w", "%{http_code}", "https://github.com"],
                   capture_output=True, text=True)
assert r.stdout.strip() == "200", "No internet. Settings -> Internet -> On"
print("internet: ok")

print("\ndisk:")
print(subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/tmp", "/root"],
                     capture_output=True, text=True).stdout)

---
## 2. Config

`WAN2GP_COMMIT` is the single most important line here. Leave it as `"main"` for the
very first run, then **paste the resolved SHA back in** — cell 6 prints it. Upstream
moves, the fp16 patches are string matches against it, and an unpinned checkout is how
you end up debugging a model that loaded "fine" and produces noise.

Set `REPO_URL` to your own remote. If you'd rather not push this repo anywhere, upload
it as a Kaggle Dataset and point `REPO_DIR` at `/kaggle/input/<dataset-name>` instead —
but note that input datasets are read-only, so generated shots must go elsewhere.

In [ ]:
import os, pathlib

# ---- edit these -----------------------------------------------------------
REPO_URL      = os.environ.get("REPO_URL", "")        # e.g. https://github.com/you/stickman.git
WAN2GP_COMMIT = "main"                                # PIN THIS after the first run
VSCODE_PASSWORD = "change-me-please"                  # tunnel is public; see cell 4
# ---------------------------------------------------------------------------

WORK       = pathlib.Path("/kaggle/working")
REPO_DIR   = WORK / "stickman"
WAN2GP_DIR = WORK / "Wan2GP"
TMP        = pathlib.Path("/kaggle/tmp")
BIN        = WORK / "bin"

# /kaggle/working is 20 GB and persists; /kaggle/tmp is bigger and does not.
# Model weights are large and re-downloadable, so they belong in tmp with a
# symlink, keeping the persistent quota for the repo and generated shots.
CKPT_REAL = TMP / "ckpts"
for d in (TMP, BIN, CKPT_REAL, WORK / "logs"):
    d.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "STICKMAN_DIR": str(REPO_DIR),
    "WAN2GP_DIR":   str(WAN2GP_DIR),
    "KREA_OUT_DIR": str(REPO_DIR / "shots"),
    "KREA_PORT":    "8711",
    "PATH":         f"{BIN}:{os.environ['PATH']}",
})

KREA_URL = "http://127.0.0.1:8711"
LOGS = WORK / "logs"
for k in ("STICKMAN_DIR", "WAN2GP_DIR", "KREA_OUT_DIR"):
    print(f"{k:14} = {os.environ[k]}")
print(f"{'WAN2GP_COMMIT':14} = {WAN2GP_COMMIT}"
      + ("   <-- PIN THIS after first run" if WAN2GP_COMMIT == "main" else ""))

---
## 3. Get the repo + install code-server

`code-server` is VS Code compiled for the browser. `cloudflared` gives it a public
HTTPS URL without any account or config.

In [ ]:
import subprocess, shutil, pathlib

def sh(cmd, **kw):
    """Run a shell command, streaming output, raising on failure."""
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=kw.pop("check", True), **kw)

# -- repo -------------------------------------------------------------------
if REPO_DIR.exists():
    print(f"{REPO_DIR} exists, pulling")
    sh(f"cd {REPO_DIR} && git pull --ff-only", check=False)
elif REPO_URL:
    sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
else:
    # No remote configured: make a bare working copy so paths still resolve.
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    (REPO_DIR / "shots").mkdir(exist_ok=True)
    print(f"!! REPO_URL not set -- created an empty {REPO_DIR}.")
    print("!! Set REPO_URL in cell 2, or attach this repo as a Kaggle Dataset")
    print("!! and copy it in:  !cp -r /kaggle/input/<name>/* {REPO_DIR}/")

# -- code-server ------------------------------------------------------------
if shutil.which("code-server"):
    print("code-server already installed")
else:
    sh("curl -fsSL https://code-server.dev/install.sh | sh")

# -- cloudflared ------------------------------------------------------------
cf = BIN / "cloudflared"
if not cf.exists():
    sh("curl -fsSL -o " + str(cf) + " "
       "https://github.com/cloudflare/cloudflared/releases/latest/download/"
       "cloudflared-linux-amd64")
    cf.chmod(0o755)

print("\ncode-server:", subprocess.run(["code-server", "--version"],
                                       capture_output=True, text=True).stdout.split("\n")[0])
print("cloudflared:", subprocess.run([str(cf), "--version"],
                                     capture_output=True, text=True).stdout.strip())

---
## 4. Launch VS Code + public tunnel

The tunnel URL is **public and unauthenticated by default** — anyone with the link gets
a shell on your session. So this starts code-server with password auth. Change
`VSCODE_PASSWORD` in cell 2 to something real before running.

In [ ]:
import subprocess, time, re, os, pathlib

if VSCODE_PASSWORD == "change-me-please":
    print("!! Using the default password. Anyone who guesses the URL gets a shell.")
    print("!! Set VSCODE_PASSWORD in cell 2 and re-run this cell.\n")

cs_log = LOGS / "code-server.log"
cf_log = LOGS / "cloudflared.log"

# Idempotent: re-running this cell shouldn't stack up processes.
subprocess.run("pkill -f 'code-server' ; pkill -f 'cloudflared tunnel'",
               shell=True, check=False)
time.sleep(2)

env = {**os.environ, "PASSWORD": VSCODE_PASSWORD}
subprocess.Popen(
    ["code-server", "--bind-addr", "127.0.0.1:8080", "--auth", "password",
     "--disable-telemetry", "--disable-update-check", str(REPO_DIR)],
    stdout=open(cs_log, "w"), stderr=subprocess.STDOUT, env=env,
)
subprocess.Popen(
    [str(BIN / "cloudflared"), "tunnel", "--url", "http://127.0.0.1:8080",
     "--no-autoupdate"],
    stdout=open(cf_log, "w"), stderr=subprocess.STDOUT,
)

# cloudflared prints the URL to its log a few seconds after start.
url = None
for _ in range(40):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", cf_log.read_text())
    if m:
        url = m.group(0)
        break

print("=" * 68)
if url:
    print(f"  VS Code:  {url}")
    print(f"  Password: {VSCODE_PASSWORD}")
    print(f"  Folder:   {REPO_DIR}")
else:
    print("  No tunnel URL yet. Check the log:")
    print(f"    !tail -30 {cf_log}")
print("=" * 68)
print("\nInstall whatever extensions you want from the Extensions panel once")
print("you're in. Terminals inside VS Code inherit this session's GPU.")

---
## 5. Checkpoint

At this point you have a working browser IDE on a GPU box, and the cells below are
optional — they only matter if you want image generation.

If the tunnel URL is dead but the notebook is alive, re-run cell 4; the URL changes
every time cloudflared restarts.

---
## 6. Wan2GP checkout

Copy the printed SHA into `WAN2GP_COMMIT` in cell 2 after this succeeds.

Kaggle's base image already carries a working torch/CUDA pair. Installing Wan2GP's
full `requirements.txt` tends to drag in a different torch and break CUDA, so this
installs only what the server actually needs on top.

In [ ]:
import subprocess

if not WAN2GP_DIR.exists():
    sh(f"git clone https://github.com/DeepBeepMeep/Wan2GP.git {WAN2GP_DIR}")
if WAN2GP_COMMIT != "main":
    sh(f"cd {WAN2GP_DIR} && git checkout --quiet {WAN2GP_COMMIT}")

sha = subprocess.run(f"cd {WAN2GP_DIR} && git rev-parse HEAD",
                     shell=True, capture_output=True, text=True).stdout.strip()

# Point Wan2GP's checkpoint dir at /kaggle/tmp so weights don't eat the 20 GB quota.
ck = WAN2GP_DIR / "ckpts"
if not ck.is_symlink():
    if ck.exists():
        sh(f"rm -rf {ck}")
    ck.symlink_to(CKPT_REAL)

# Deliberately NOT `pip install -r requirements.txt` -- see the note above.
sh("pip install -q --no-cache-dir "
   "mmgp fastapi uvicorn pydantic safetensors einops omegaconf ftfy "
   "'huggingface_hub>=0.24' 'transformers>=4.44' accelerate diffusers")

print("\n" + "=" * 68)
print(f"  Wan2GP HEAD = {sha}")
print(f"  -> paste into cell 2:  WAN2GP_COMMIT = \"{sha}\"")
print("=" * 68)

---
## 7. Probe before loading

Cheap sanity check that runs in seconds and tells you whether the fp16 patches still
match this checkout, without waiting through a multi-minute model load to find out.

`!! NO MATCH` here is a **warning, not a blocker** — `krea_server.py` sweeps any
surviving bf16 tensors after load, so generation stays correct. It just means a slower
load, and it's your cue to fix the find-strings in `scripts/patches/t4_fp16.json`.

In [ ]:
!cd {REPO_DIR} && python scripts/krea_server.py --probe 2>&1 | tail -60

---
## 8. Start the model server

Loads Krea 2 Turbo Edit once and holds it on GPU0. First run also downloads the
weights, so expect **10-20 minutes**; later runs in the same session are ~3-5 minutes.

The server is plain HTTP on `127.0.0.1:8711`, which means anything inside VS Code —
a terminal, a script, an extension — can drive it.

In [ ]:
import subprocess, time, json, urllib.request, os

srv_log = LOGS / "krea_server.log"
subprocess.run("pkill -f krea_server.py", shell=True, check=False)
time.sleep(2)

subprocess.Popen(
    ["python", str(REPO_DIR / "scripts" / "krea_server.py")],
    stdout=open(srv_log, "w"), stderr=subprocess.STDOUT,
    env={**os.environ}, cwd=str(REPO_DIR),
)

def health():
    try:
        with urllib.request.urlopen(f"{KREA_URL}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

# Poll instead of sleeping a fixed interval: load time varies with download state.
print("waiting for model load (tailing log)...")
deadline = time.time() + 25 * 60
last = 0
while time.time() < deadline:
    h = health()
    if h and h.get("status") == "ready":
        print("\n" + json.dumps(h, indent=2))
        break
    if time.time() - last > 30:
        last = time.time()
        tail = subprocess.run(f"tail -3 {srv_log}", shell=True,
                              capture_output=True, text=True).stdout.strip()
        print(tail or "  (no output yet)")
    if not srv_log.exists() or "FATAL" in srv_log.read_text()[-4000:]:
        print(subprocess.run(f"tail -40 {srv_log}", shell=True,
                             capture_output=True, text=True).stdout)
        raise SystemExit("server died during load -- see the log above")
    time.sleep(10)
else:
    raise SystemExit(f"timed out. Check: !tail -50 {srv_log}")

---
## 9. Smoke test

Two generations: one text-only, one using a real keyframe from `frames/` as a style
reference. If the second looks nothing like the reference, `video_prompt_type` isn't
reaching the model — check cell 7's probe output for the generate signature.

At 1024x576 / 8 steps, expect roughly **20-25 s** per image on a T4.

In [ ]:
import json, urllib.request, time
from IPython.display import Image as IPyImage, display

def generate(prompt, out_name, refs=(), ref_mode="KI", **kw):
    body = {"prompt": prompt, "out_name": out_name,
            "ref_paths": list(refs), "ref_mode": ref_mode, **kw}
    req = urllib.request.Request(
        f"{KREA_URL}/generate", data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=600) as r:
        return json.load(r)

STYLE = ("minimal hand-drawn stickman on a pure white background, thick 6px solid "
         "black monoline ink strokes, flat saturated color fills, no gradients, no shadows, "
         "whiteboard explainer illustration")

# 1. text only
r1 = generate(f"{STYLE}. A stickman sitting on a chair sweating beside a desk fan, "
              "blue airflow curves, red squiggly heat lines radiating outward.",
              "smoke_01_text.png")
print(json.dumps(r1, indent=2))
display(IPyImage(filename=r1["path"], width=512))

# 2. with a channel keyframe as the style reference
r2 = generate(f"{STYLE}. A stickman with a mechanical engine visible inside his "
              "torso, red heat waves, a semicircular gauge with the needle in the "
              "red zone.",
              "smoke_02_ref.png",
              refs=["frames/01_fan_death/04_scientific_cross_section_120s.jpg"])
print(json.dumps(r2, indent=2))
display(IPyImage(filename=r2["path"], width=512))

print("\ntiming:", json.loads(urllib.request.urlopen(f"{KREA_URL}/stats").read()))

---
## 10. Keepalive

Kaggle reclaims idle sessions. Run this in the background and leave the tab open;
work happens in the VS Code tab, not here.

Interrupt the kernel to stop it.

In [ ]:
import time, json, urllib.request, datetime

while True:
    try:
        h = json.loads(urllib.request.urlopen(f"{KREA_URL}/health", timeout=5).read())
        state = f"{h['status']} | {h.get('generated', 0)} generated | {h.get('vram_free_gb')} GB free"
    except Exception as e:
        state = f"server unreachable ({type(e).__name__})"
    print(f"{datetime.datetime.now():%H:%M:%S}  {state}", flush=True)
    time.sleep(300)

---
## Troubleshooting

```python
!tail -50 {LOGS}/krea_server.log      # model load / generation errors
!tail -30 {LOGS}/cloudflared.log      # tunnel URL, disconnects
!tail -30 {LOGS}/code-server.log      # editor startup
!nvidia-smi                           # who's holding VRAM
```

| Symptom | Cause | Fix |
|---|---|---|
| Tunnel URL 502s | code-server died | Re-run cell 4 |
| URL changed | cloudflared restarted | Re-run cell 4, take the new URL |
| `!! NO MATCH` in probe | upstream drifted | Fix find-strings in `scripts/patches/t4_fp16.json`; the fp16 sweep covers you meanwhile |
| Large `fp16_sweep.converted` | patches not matching | Same as above — correct output, slow load |
| References ignored | `video_prompt_type` not reaching generate | Check the generate signature in cell 7's probe |
| CUDA OOM | something else on GPU0 | `!nvidia-smi`; restart the server cell |
| `503 model still loading` | load not finished | Poll `/health` until `ready` |

## Getting shots back

The tunnel is an editor, not a file server. To pull generated images out:

```bash
# from a VS Code terminal
cd /kaggle/working/stickman
git add shots/final && git commit -m "shots" && git push
```

Or right-click → Download in the VS Code file explorer for one-offs. Anything left
only in `/kaggle/tmp` is gone when the session ends.